# IMDb animation series 2022: custom score

Reads `data.csv` (the full scrape, ~1.27 million rows, **not in this repo**). If it is missing, falls back to `data_sample.csv`, a small sample recovered from the `*_x` columns of the committed `out.csv`. **With the sample, the means/maxima (`vm, rm, pm, pM, vM`) differ from the full-data ones, so `cal` values differ from the committed `out.csv`.** Results go to `out.csv` for the full data and `out_sample.csv` for the sample.

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
FULL = os.path.exists('data.csv')
alldata = pd.read_csv('data.csv' if FULL else 'data_sample.csv', dtype=str)
series = pd.read_csv('series22.csv', dtype=str)
print('USING FULL data.csv' if FULL else 'USING SMALL data_sample.csv: numbers are NOT comparable to the full-data out.csv')
print(len(alldata), 'rows in alldata;', len(series), 'rows in series22')

USING SMALL data_sample.csv: numbers are NOT comparable to the full-data out.csv
393 rows in alldata; 294 rows in series22


Clean: strip thousands separators and trailing dots, convert to numbers (bad values become NaN and are dropped).

In [3]:
for df in (alldata, series):
    for col in ('place', 'vote', 'rate'):
        df[col] = pd.to_numeric(df[col].str.replace(',', '').str.rstrip('.'), errors='coerce')
alldata = alldata.dropna(subset=['place', 'vote', 'rate']).reset_index(drop=True)
series = series.dropna(subset=['place', 'vote', 'rate']).reset_index(drop=True)
alldata.dtypes

place      int64
title        str
rate     float64
vote       int64
dtype: object

Statistics: means (`vm, rm, pm`) and maxima (`pM, vM`) of votes, rating and place.

In [4]:
vm = alldata['vote'].mean()
rm = alldata['rate'].mean()
pm = alldata['place'].mean()
pM = alldata['place'].max()
vM = alldata['vote'].max()
print(len(alldata), vm, rm, pm, pM, vM)

393 1208.7506361323155 7.0002544529262085 133510.4554707379 635506 47530


Custom score `cal`: the original seven-term formula, vectorized (the original loops/`apply` cells were slow and ran out of memory on the full data).

In [5]:
v, p, r = alldata['vote'], alldata['place'], alldata['rate']
vp = v + (pM - p)
alldata['cal'] = (vp / (vp + vm + pm) * r
                  + (vm + pm) / (vp + vm + pm) * rm
                  + r / (r + rm) * (vp / (pM + vM) * 10)
                  + rm / (r + rm) * ((vm + pm) / (vM + pM) * 10)
                  + v / (v + vm) * ((pM - p) / pM * 10)
                  + vm / (v + vm) * (pm / pM * 10)
                  + (pM - p) / ((pM - p) + pm) * (v / vM * 10)
                  + pm / ((pM - p) + pm) * (v / vM * 10))
alldata.head()

,place,title,rate,vote,cal
0,124,Chainsaw Man,8.7,19865,28.329954
1,311,Sonic Prime,7.3,926,18.687299
2,336,Bleach: Thousand-Year Blood War,9.2,13835,27.341367
3,764,Spy x Family,8.5,24625,29.229958
4,802,Cyberpunk: Edgerunners,8.3,47530,34.198237


Join with `series22.csv` on title (the committed data has no IMDb id; if both files have an `imdb_id` column, that is used instead). Titles that occur more than once on either side are ambiguous (different shows share a name), so they are dropped instead of producing duplicate/mismatched rows.

In [6]:
key = 'imdb_id' if 'imdb_id' in alldata.columns and 'imdb_id' in series.columns else 'title'
a = alldata[~alldata[key].duplicated(keep=False)]
s = series[~series[key].duplicated(keep=False)]
print('dropped ambiguous', key, 'rows:', len(alldata) - len(a), 'from alldata,', len(series) - len(s), 'from series22')
com = pd.merge(a, s, how='inner', on=key, suffixes=('_x', '_y'))
com = com.sort_values(by='cal', ascending=False)
com.head(10)

dropped ambiguous title rows: 126 from alldata, 2 from series22


,place_x,title,rate_x,vote_x,cal,place_y,rate_y,vote_y
3,802,Cyberpunk: Edgerunners,8.3,47530,34.198237,4,8.3,47639
5,945,The Legend of Vox Machina,8.4,25233,29.261623,7,8.4,25257
2,764,Spy x Family,8.5,24625,29.229958,3,8.5,24677
15,4807,I Am Groot,6.7,29170,28.233492,21,6.7,29218
8,1695,Tales of the Jedi,8.3,20876,28.114809,8,8.3,20952
1,336,Bleach: Thousand-Year Blood War,9.2,13835,27.341367,2,9.2,13923
18,6023,The Boys Presents: Diabolical,6.9,13306,24.636900,28,6.9,13316
14,4625,My Dress-Up Darling,8.1,8352,24.538027,17,8.1,8370
17,5455,The Cuphead Show!,7.4,8268,23.748626,24,7.4,8275
36,11141,Baymax!,7.1,6979,22.868720,44,7.1,6990


In [7]:
out = 'out.csv' if FULL else 'out_sample.csv'
com.to_csv(out, index=False)
print(len(com), 'rows written to', out)

267 rows written to out_sample.csv
